In [9]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from kilosort import io
from kilosort.data_tools import (
    mean_waveform, cluster_templates, get_good_cluster, get_cluster_spikes,
    get_spike_waveforms, get_best_channels, get_best_channel
    )

# Indicate where sorting results were saved
results_dir = Path(r'D:\Data\Kelton\probe_data\KW116\KW116_09282026_rec_D1_RLat1_g0\kilosort4\sorter_output')
binfile     = Path(r'D:\Data\Kelton\probe_data\KW116\KW116_09282026_rec_D1_RLat1_g0\KW116_09282026_rec_D1_RLat1_g0_imec0\KW116_09282026_rec_D1_RLat1_g0_t0.imec0.ap.bin')
backup_dir = results_dir.parent

master_csv_path = results_dir / 'all_clusters_master_waveforms.csv'


In [13]:
# Load original ops
ops = io.load_ops(results_dir / 'ops.npy')

# Save a backup of the original
io.save_ops(ops, results_dir = backup_dir)

# Overwrite the broken ops path with the correct path
ops['filename'] = binfile
ops['data_dir'] = binfile.parent
ops['n_chan_bin'] = 256

# Save it back to disk so Kilosort's internal functions read it correctly
io.save_ops(ops, results_dir = results_dir)
print("Fixed path references and channel count inside ops.npy!")

Fixed path references and channel count inside ops.npy!


In [14]:
# Load channel map to ensure we align the headers with correct channel indices
channel_map = np.load(results_dir / 'channel_map.npy').flatten()
# Format the header string: "Channel_0,Channel_1,Channel_2..."
channel_headers = ",".join([f"Channel_{ch}" for ch in channel_map])

# Load all unique cluster IDs from the sorting output
spike_clusters  = np.load(results_dir / 'spike_clusters.npy')
all_cluster_ids = np.unique(spike_clusters)
spike_templates = np.load(results_dir / 'spike_templates.npy')

print(f"Found {len(all_cluster_ids)} total clusters to process.")


Found 593 total clusters to process.


In [15]:
# Find all unique cluster IDs
all_cluster_ids = np.unique(spike_clusters)
print(f"Processing {len(all_cluster_ids)} clusters into a Timepoints x Clusters master file...")

# Dictionary to collect {Column_Label: 1D_Waveform_Array}
waveform_dict = {}
cluster_ids_meta = []
physical_channels_meta = []

# Loop through every cluster to gather waveforms and metadata
for cluster_id in all_cluster_ids:
    try:
        # Extract the mean waveform matrix (best=True returns the single best channel's waveform)
        mean_wv, spike_subset = mean_waveform(cluster_id, results_dir, n_spikes=100,
                                              bfile=None, best=True)
        
        # Validate data presence
        if mean_wv is None or len(spike_subset) == 0:
            continue
            
        # Ensure it's flattened to a 1D vector of timepoints
        mean_wv_flat = mean_wv.flatten()
        
        # Find which physical channel this cluster belongs to  
        cluster_spikes_templates = spike_templates[spike_clusters == cluster_id]
        if len(cluster_spikes_templates) > 0:
            physical_channel = get_best_channel(results_dir, cluster_id)
        else:
            physical_channel = -1 # Placeholder if untrackable

        # Save metadata and data array using a standardized column label (e.g., "Cluster_0")
        col_label = f"Cluster_{cluster_id}"
        waveform_dict[col_label] = mean_wv_flat
        cluster_ids_meta.append(cluster_id)
        physical_channels_meta.append(physical_channel)
        
        print(f"Processed Cluster {cluster_id} (Physical Channel: {physical_channel})")
        
    except Exception as e:
        print(f"Failed to extract cluster {cluster_id}. Error: {e}")

# Format and construct the final dataframe
if waveform_dict:
    print("\nStructuring data matrix into Timepoints x Clusters format...")
    # Create the core data dataframe (Rows = Timepoints, Columns = Clusters)
    df_waves = pd.DataFrame(waveform_dict)
    
    # Insert Timepoint index column at the front
    df_waves.insert(0, 'Timepoint', np.arange(len(df_waves)))
    
    # Build the metadata rows matching the exact columns of df_waves
    # Timepoint column gets text labels for metadata rows
    meta_rows = [
        ['cluster_id'] + cluster_ids_meta,
        ['physical_channel'] + physical_channels_meta
    ]
    
    # Create a metadata DataFrame with matching column names
    df_meta = pd.DataFrame(meta_rows, columns=df_waves.columns)
    
    # Stack the metadata dataframes on top of the waveform dataframe
    master_df = pd.concat([df_meta, df_waves], ignore_index=True)
    
    print(f"Writing master file to disk (Shape: {master_df.shape})...")
    # Save to CSV without headers or index numbers to preserve the custom multi-row layout precisely
    master_df.to_csv(master_csv_path, index=False, header=False)
    
    print(f"Success! Master spreadsheet generated at:\n  {master_csv_path}")
else:
    print("Process finished, but no valid data could be aggregated.")

Processing 593 clusters into a Timepoints x Clusters master file...
Failed to extract cluster 0. Error: Bytes in binary file did not divide evenly, incorrect n_chan_bin ('number of channels' in GUI).
Failed to extract cluster 1. Error: Bytes in binary file did not divide evenly, incorrect n_chan_bin ('number of channels' in GUI).
Failed to extract cluster 2. Error: Bytes in binary file did not divide evenly, incorrect n_chan_bin ('number of channels' in GUI).
Failed to extract cluster 3. Error: Bytes in binary file did not divide evenly, incorrect n_chan_bin ('number of channels' in GUI).
Failed to extract cluster 4. Error: Bytes in binary file did not divide evenly, incorrect n_chan_bin ('number of channels' in GUI).
Failed to extract cluster 5. Error: Bytes in binary file did not divide evenly, incorrect n_chan_bin ('number of channels' in GUI).
Failed to extract cluster 6. Error: Bytes in binary file did not divide evenly, incorrect n_chan_bin ('number of channels' in GUI).
Failed t

KeyboardInterrupt: 